# Swiss Legal Citation Retrieval — Full Pipeline

**Steps:**
1. Clone repo & install deps
2. Download Kaggle data
3. Download BGE reranker from HuggingFace
4. Prepare training data (with hard negatives)
5. Fine-tune embedding model
6. Calibrate retrieval threshold on val
7. Rerank with cross-encoder
8. (Optional) BM25 hybrid
9. Download submission

> **GPU:** Set Runtime → Change runtime type → T4 GPU before running

## 0. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('VRAM (GB)  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: No GPU — training will be very slow. Enable GPU in Runtime settings.')

## 1. Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/farhanwew/LLM---Document-Retreival.git'
REPO_DIR = '/content/llm-legal'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Pull latest changes if already cloned
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!git log --oneline -3

In [ ]:
!pip install -q \
    sentence-transformers>=3.0 \
    transformers>=4.40 \
    datasets \
    rank_bm25 \
    kaggle \
    huggingface_hub

print('Dependencies installed.')

## 2. Download Kaggle Data

Upload your `kaggle.json` API token first (from kaggle.com → Account → API → Create New Token).

In [ ]:
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()

# Move to expected location
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('kaggle.json configured.')

In [ ]:
COMPETITION = 'llm-agentic-legal-information-retrieval'

!mkdir -p data
!kaggle competitions download -c {COMPETITION} -p data/
!cd data && unzip -o {COMPETITION}.zip && rm {COMPETITION}.zip

!echo 'Downloaded files:'
!ls -lh data/

## 3. Download BGE Reranker from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download

RERANKER_ID   = 'BAAI/bge-reranker-v2-m3'
RERANKER_PATH = 'finetune/models/bge-reranker-v2-m3'

if not os.path.exists(RERANKER_PATH):
    print(f'Downloading {RERANKER_ID} ...')
    snapshot_download(
        repo_id=RERANKER_ID,
        local_dir=RERANKER_PATH,
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
    )
    print('Done.')
else:
    print(f'Reranker already exists at {RERANKER_PATH}')

!ls -lh {RERANKER_PATH}/

## 4. Prepare Training Data

Builds `(query, positive_doc, hard_negative_doc)` triplets.  
Hard negatives are mined from the corpus using the base embedding model.

**Modes:**
- `original` — train queries in DE/FR/IT as-is  
- `translated` — translate all queries to English  
- `both` — both (doubles training data, recommended)

In [ ]:
# Config — adjust if needed
QUERY_MODE = 'both'    # 'original' | 'translated' | 'both'
MODEL_TYPE = 'e5-large'  # 'e5-large' | 'gemma'
RUN_NAME   = f'scenario-{QUERY_MODE}'

!mkdir -p finetune/logs

!python finetune/01_prepare_data.py \
    --query-mode {QUERY_MODE} \
    --model {MODEL_TYPE} \
    --log-file finetune/logs/prepare_{MODEL_TYPE}_{QUERY_MODE}.txt

print('\n--- Log tail ---')
!tail -20 finetune/logs/prepare_{MODEL_TYPE}_{QUERY_MODE}.txt

In [ ]:
# Verify dataset has the 'negative' column (hard negatives)
from datasets import load_from_disk

ds = load_from_disk(f'finetune/prepared_data/{MODEL_TYPE}/{QUERY_MODE}')
print('Train size :', len(ds['train']), 'rows')
print('Eval size  :', len(ds['eval']), 'rows')
print('Columns    :', ds['train'].column_names)
assert 'negative' in ds['train'].column_names, 'ERROR: negative column missing! Check mining step.'
print('\nSample:')
print('  anchor  :', ds['train']['anchor'][0][:100])
print('  positive:', ds['train']['positive'][0][:100])
print('  negative:', ds['train']['negative'][0][:100])

## 5. Fine-tune Embedding Model

Trains with `MultipleNegativesRankingLoss` using hard negative triplets.  
Best checkpoint selected by **Macro F1** (competition metric).

In [ ]:
import subprocess, sys, os, time, re

log_file = f'finetune/logs/{RUN_NAME}.txt'
os.makedirs('finetune/logs', exist_ok=True)

cmd = [
    sys.executable, '-u', 'finetune/02_finetune.py',
    '--query-mode', QUERY_MODE,
    '--model', MODEL_TYPE,
    '--run-name', RUN_NAME,
    '--log-file', log_file,
]

tqdm_pat = re.compile(r'^\s*\d+%|it/s\]|Writing model shards|Materializing')

with open(log_file, 'w') as log, \n     subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as proc:
    for line in proc.stdout:
        log.write(line)
        log.flush()
        if not tqdm_pat.search(line):  # skip tqdm bars
            print(line, end='', flush=True)

rc = proc.wait()
print(f'
Done (exit code {rc})')


In [ ]:
# Check final model was saved
import os
final_path = f'finetune/models/{RUN_NAME}/final'
assert os.path.exists(final_path), f'Model not found at {final_path}'
print('Model saved at:', final_path)
!ls -lh {final_path}/

## 6. Evaluate on val.csv

In [ ]:
!python finetune/03_evaluate.py \
    --models {RUN_NAME} \
    --model-type {MODEL_TYPE} \
    --top-k 20 \
    --log-file finetune/logs/evaluate.txt

!cat finetune/logs/evaluate.txt | grep -A 20 'RESULTS COMPARISON'

## 7. Calibrate Retrieval Threshold on Val

Sweeps cosine similarity thresholds to find the best **Macro F1** on `val.csv`.  
This avoids the fixed top-K precision penalty.

In [ ]:
!python finetune/04_inference.py \
    --model {RUN_NAME} \
    --model-type {MODEL_TYPE} \
    --sweep-val \
    --no-court \
    --log-file finetune/logs/threshold_sweep.txt

!cat finetune/logs/threshold_sweep.txt | grep -A 30 'Threshold sweep'

In [ ]:
# Set the best threshold from the sweep above
SCORE_THRESHOLD = 0.78   # <-- update this from sweep output above
print(f'Using score_threshold = {SCORE_THRESHOLD}')

## 8. Generate Bi-encoder Submission (baseline)

In [ ]:
!python finetune/04_inference.py \
    --model {RUN_NAME} \
    --model-type {MODEL_TYPE} \
    --score-threshold {SCORE_THRESHOLD} \
    --no-court \
    --output submission_biencoder.csv \
    --log-file finetune/logs/inference_biencoder.txt

import pandas as pd
sub = pd.read_csv('submission_biencoder.csv')
print(f'Rows: {len(sub)}')
print(sub.head(5).to_string(index=False))

## 9. Calibrate Reranker Threshold on Val

BGE reranker returns **raw logits** (not probabilities, range ~-10 to +10).  
Must calibrate threshold on val before using on test.

In [ ]:
!python finetune/05_rerank.py \
    --bi-encoder-model {RUN_NAME} \
    --reranker-path {RERANKER_PATH} \
    --model-type {MODEL_TYPE} \
    --rerank-top-n 50 \
    --sweep-val \
    --no-court \
    --log-file finetune/logs/rerank_sweep.txt

!cat finetune/logs/rerank_sweep.txt | grep -A 30 'threshold sweep'

In [ ]:
# Set the best reranker threshold from sweep above (raw logit value)
RERANKER_THRESHOLD = 0.0   # <-- update from sweep output above
print(f'Using reranker score_threshold = {RERANKER_THRESHOLD}')

## 10. Generate Reranked Submission (best result)

In [ ]:
!python finetune/05_rerank.py \
    --bi-encoder-model {RUN_NAME} \
    --reranker-path {RERANKER_PATH} \
    --model-type {MODEL_TYPE} \
    --rerank-top-n 50 \
    --score-threshold {RERANKER_THRESHOLD} \
    --no-court \
    --output submission_reranked.csv \
    --log-file finetune/logs/rerank_final.txt

sub = pd.read_csv('submission_reranked.csv')
print(f'Rows: {len(sub)}')
print(sub.head(5).to_string(index=False))

## 11. (Optional) BM25 Hybrid

Combines BM25 keyword matching with dense retrieval via Reciprocal Rank Fusion.  
Helps for queries with specific legal article numbers like `Art. 42 OR`.

In [ ]:
# First check if BM25 hybrid helps on val
!python finetune/06_hybrid.py \
    --bi-encoder-model {RUN_NAME} \
    --model-type {MODEL_TYPE} \
    --reranker-path {RERANKER_PATH} \
    --sweep-val \
    --no-court \
    --log-file finetune/logs/hybrid_sweep.txt

!cat finetune/logs/hybrid_sweep.txt | grep -A 10 'Val comparison'

In [ ]:
# Only run if val comparison above shows RRF > Dense
!python finetune/06_hybrid.py \
    --bi-encoder-model {RUN_NAME} \
    --model-type {MODEL_TYPE} \
    --reranker-path {RERANKER_PATH} \
    --score-threshold {RERANKER_THRESHOLD} \
    --no-court \
    --output submission_hybrid.csv \
    --log-file finetune/logs/hybrid_final.txt

sub = pd.read_csv('submission_hybrid.csv')
print(f'Rows: {len(sub)}')
print(sub.head(5).to_string(index=False))

## 12. Download Submissions

In [ ]:
from google.colab import files

# Download whichever submission files exist
for fname in ['submission_biencoder.csv', 'submission_reranked.csv', 'submission_hybrid.csv']:
    if os.path.exists(fname):
        print(f'Downloading {fname}...')
        files.download(fname)
    else:
        print(f'Not found: {fname} (skipping)')

## (Optional) Save Fine-tuned Model to Drive

In [ ]:
# Mount Google Drive to persist the fine-tuned model across sessions
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_PATH = f'/content/drive/MyDrive/swiss-legal/{RUN_NAME}'
!mkdir -p {DRIVE_SAVE_PATH}
!cp -r finetune/models/{RUN_NAME}/final {DRIVE_SAVE_PATH}/
print(f'Model saved to {DRIVE_SAVE_PATH}/final')

In [ ]:
# To restore model from Drive in a new session:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_SAVE_PATH = '/content/drive/MyDrive/swiss-legal/scenario-both'
# !mkdir -p finetune/models/scenario-both
# !cp -r {DRIVE_SAVE_PATH}/final finetune/models/scenario-both/final
# Then skip straight to Step 7 (threshold calibration)